# Lesson 9: Closed loop

**You will learn:** how a subject acts in an environment, where each action changes the next observation.
**Requires:** CPU only, no downloads, under a minute.

In the earlier lessons the inputs were fixed in advance. In a closed-loop task, such as a game or a robot task, the subject's action decides what it sees next. Brain-Score uses the same session idea:

| Object | Meaning |
| --- | --- |
| Environment | Your simulator or game, with `reset()` and `step(action)`. |
| `EnvironmentStep` | One observation, plus episode signals such as `is_first`, `is_last` and `reward`. |
| `EnvironmentResponse` | The subject's action for that step. |
| `EnvironmentSession` | A session whose next input depends on the last response. |
| `policy` | The `BrainScoreModel` part that maps an `EnvironmentStep` to an `EnvironmentResponse`. |

In [1]:
import numpy as np
from brainscore_core import EnvironmentStep, EnvironmentResponse

class Corridor:
    """A five-cell corridor. The agent starts at 0 and the goal is cell 4."""

    def reset(self, seed=None, options=None):
        self.position = 0
        return EnvironmentStep(observation={'position': 0}, step_num=0, is_first=True)

    def step(self, action):
        self.position = max(0, self.position + int(action))
        done = self.position >= 4
        return EnvironmentStep(observation={'position': self.position}, step_num=self.position,
                               is_last=done, is_terminal=done, reward=float(done))

    def close(self):
        pass

## A policy

A policy is a plain function. This one always moves right. A real policy would be a trained network or a robot controller.

In [2]:
from brainscore_core import BrainScoreModel

def move_right(step):
    return EnvironmentResponse(action=np.array(1))

agent = BrainScoreModel(identifier='move-right', policy=move_right)
print('inputs: ', agent.in_channels)
print('outputs:', agent.out_channels)

inputs:  {'observation'}
outputs: {'motor'}


## Run one episode

`EnvironmentSession` resets the environment, delivers each observation, applies each action, and stops when the episode ends. The subject participates exactly as in Lesson 1.

In [3]:
from brainscore_core.streaming_helpers import EnvironmentSession

with EnvironmentSession(Corridor()) as session:
    agent.participate(session)
    print('episode complete:', session.complete)
    for observation, action in zip(session.input_events, session.collect('motor')):
        print(observation.payload.observation, '->', action.action)
agent.reset()

episode complete: True
{'position': 0} -> 1
{'position': 1} -> 1
{'position': 2} -> 1
{'position': 3} -> 1


The `run_environment(subject, environment)` helper does the same in one call and returns the applied actions.

## Summary

- A closed-loop task is still a session; only the session type changes.
- The `policy` part maps each `EnvironmentStep` to an `EnvironmentResponse`.
- Existing robotics evaluators (for example LIBERO or DROID) can keep their own loop; see the docs.

**Next:** [Lesson 10](10_run_an_experiment.ipynb) organizes conditions, repetitions, recording and perturbation into a reproducible experiment.